## Setup

In [ ]:
import os
import optuna
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics import average_precision_score
from optuna.integration import XGBoostPruningCallback
import cupy as cp

from src.py_src.models import GatekeeperModel, GreatFilterModel
import joblib

In [ ]:
load_dotenv()

XRAY_SLIDED_PATH = os.path.join(os.getenv("SLIDED_PATH"), "xray_slided.parquet")

xray_slided_df = pd.read_parquet(XRAY_SLIDED_PATH)

target_class = 'target_class_in_24h'
target_flux = 'target_flux_in_24h'
metadata_cols = ['run_id', 'time']

features = [col for col in xray_slided_df.columns if col not in metadata_cols + [target_class, target_flux]]

# Isolamento conceitual das features (X) e do alvo bruto (y)
X = xray_slided_df[features]
y = xray_slided_df[target_class]

In [ ]:
xray_slided_df.head()

## Preparing Data (Block-Chronological Split)

In [ ]:
cols_to_keep = features + ['time', target_class, target_flux]
gatekeeper_pool = xray_slided_df[cols_to_keep].copy()

def block_chronological_split(df, time_col, target_col, flux_col, _train_years, _val_years, _test_years, purge_hours=24):
    """
    Divide os dados em blocos baseados em anos para distribuir as fases do Ciclo Solar em todos os conjuntos.
    Aplica um Purge Gap Simétrico na transição de blocos:
    Remove 'purge_hours' do final do bloco anterior (para isolar o Target que olha pro futuro)
    E 'purge_hours' do início do novo bloco (para isolar as Features que olham pro passado).
    """
    df = df.sort_values(time_col).reset_index(drop=True).copy()
    df['year'] = df[time_col].dt.year

    # Atribuição inicial aos blocos
    df['split'] = 'none'
    df.loc[df['year'].isin(_train_years), 'split'] = 'train'
    df.loc[df['year'].isin(_val_years), 'split'] = 'val'
    df.loc[df['year'].isin(_test_years), 'split'] = 'test'

    # Remove amostras descartadas
    df = df[df['split'] != 'none'].reset_index(drop=True)

    # Identifica pontos de transição de bloco na linha do tempo
    df['block_change'] = df['split'] != df['split'].shift(1)
    df.loc[0, 'block_change'] = False # A primeira linha não é uma transição

    drop_indices = set()
    change_indices = df[df['block_change']].index
    purge_td = pd.Timedelta(hours=purge_hours)

    for idx in change_indices:
        transition_time = df.loc[idx, time_col]
        # Purga Simétrica: 'purge_hours' para trás e 'purge_hours' para frente
        start_purge = transition_time - purge_td
        end_purge = transition_time + purge_td

        to_drop = df[(df[time_col] >= start_purge) & (df[time_col] < end_purge)].index
        drop_indices.update(to_drop)

    df_purged = df.drop(index=list(drop_indices)).copy()

    dict_ = {'x': {}, 'y': {}, 'flux': {}}
    cols_to_drop = [target_col, flux_col, time_col, 'year', 'split', 'block_change']

    for split_name in ['train', 'val', 'test']:
        split_df = df_purged[df_purged['split'] == split_name].copy()
        dict_['x'][split_name] = split_df.drop(columns=cols_to_drop, errors='ignore')
        dict_['y'][split_name] = split_df[target_col].apply(lambda lb: 1 if lb >= 3 else 0)
        dict_['flux'][split_name] = split_df[flux_col]

    return dict_

# TREINO: A maior parte do Ciclo Solar 24 (Ascensão, Mínimo e Máximo misturados)
train_years = [2010, 2011, 2013, 2014, 2015, 2016, 2018, 2019]

# VALIDAÇÃO: Amostragem representativa do Ciclo 24 para o Optuna
# Pegamos 2012 (subida/atividade alta) e 2017 (descida/atividade baixa)
val_years = [2012, 2017]

# TESTE: O Ciclo Solar 25 inteiro (Isolado e intocado)
test_years = [2020, 2021, 2022, 2023, 2024]

data = block_chronological_split(
    df=gatekeeper_pool,
    time_col='time',
    target_col=target_class,
    flux_col=target_flux,
    _train_years=train_years,
    _val_years=val_years,
    _test_years=test_years,
    purge_hours=24
)

print(f"Tamanho do Treino (Original): {len(data['x']['train'])} amostras")
print(f"Tamanho da Validação (Original): {len(data['x']['val'])} amostras")
print(f"Tamanho do Teste (Original): {len(data['x']['test'])} amostras")

## 🛑 Nota Metodológica: Viés de Exposição no Treinamento (Exposure Bias)

Para treinar este especialista (*Great Filter*), utilizamos a abordagem de "Realidade Suja": o modelo será alimentado exclusivamente com as instâncias que sobreviveram à filtragem do **Gatekeeper**, incluindo os Falsos Positivos autênticos. 

**Paradoxo da Confiança no Treino:** Ao passarmos o conjunto de Treinamento (`X_train`) pelo Gatekeeper, o modelo anterior emitirá previsões sobre dados que ele já "memorizou" na sua etapa de fit. Consequentemente, a taxa de Falsos Positivos no conjunto de Treino filtrado será menor do que no ambiente de produção. Aceitamos esse *covariate shift* matematicamente. A validação real do poder de filtragem e ajuste de hiperparâmetros deste especialista recairá sobre o conjunto de Validação (`X_val`), que permanece estritamente cego (Out-of-Sample) em relação ao treinamento do Gatekeeper.

In [ ]:
gatekeeper_path = os.path.join(os.getenv('GLOBAL_XRAY_FINAL_MODELS_PATH'), 'gatekeeper_v1.joblib')
gatekeeper = GatekeeperModel.load(gatekeeper_path)
print(f"Limiar de corte embutido no Gatekeeper: {gatekeeper.threshold:.4f}\n")

In [ ]:
# A função predict() do Gatekeeper já utiliza o threshold calibrado automaticamente. 
# Aplicamos a máscara sobre as 23 features originais (data['x']), NÃO sobre as features filtradas. 
# Assim, o Great Filter terá todas as opções disponíveis para sua própria etapa de Discovery.
def filter_by_gatekeeper(x_raw: pd.DataFrame, y_raw: pd.Series, flux_raw: pd.Series):
    """Filtra o dataset mantendo apenas as instâncias que o Gatekeeper classificou como Alerta (1)."""
    mask = gatekeeper.predict(x_raw) == 1
    
    x_survived = x_raw[mask].copy()
    y_survived = y_raw[mask].copy()
    flux_survived = flux_raw[mask].copy()
    
    return x_survived, y_survived, flux_survived

# Substituindo os conjuntos de dados pela versão "sobrevivente"
X_train_gf, y_train_gf, flux_train_gf = filter_by_gatekeeper(data['x']['train'], data['y']['train'], data['flux']['train'])
X_val_gf, y_val_gf, flux_val_gf = filter_by_gatekeeper(data['x']['val'], data['y']['val'], data['flux']['val'])
X_test_gf, y_test_gf, flux_test_gf = filter_by_gatekeeper(data['x']['test'], data['y']['test'], data['flux']['test'])

def print_funnel_report(name: str, original_y: pd.Series, survived_y: pd.Series):
    """Gera um relatório detalhado do impacto da filtragem na distribuição de classes."""
    orig_total = len(original_y)
    surv_total = len(survived_y)
    red_pct = ((orig_total - surv_total) / orig_total) * 100
    
    orig_pos = (original_y == 1).sum()
    orig_neg = (original_y == 0).sum()
    surv_pos = (survived_y == 1).sum()
    surv_neg = (survived_y == 0).sum()
    
    noise_reduction = ((orig_neg - surv_neg) / orig_neg * 100) if orig_neg > 0 else 0
    signal_retention = (surv_pos / orig_pos * 100) if orig_pos > 0 else 0
    
    ratio_orig = orig_neg / orig_pos if orig_pos > 0 else 0
    ratio_surv = surv_neg / surv_pos if surv_pos > 0 else 0
    
    print(f"📊 {name.upper()} FUNNEL REPORT")
    print(f"   Volume Total:  {orig_total} -> {surv_total} (Redução global de {red_pct:.1f}%)")
    print(f"   Calmaria (0):  {orig_neg} -> {surv_neg} amostras (Ruído eliminado: {noise_reduction:.1f}%)")
    print(f"   Flares (1):    {orig_pos} -> {surv_pos} amostras (Sinal retido: {signal_retention:.1f}%)")
    print(f"   Novo Balanço:  1 Positivo para cada {ratio_surv:.1f} Negativos (Original era 1:{ratio_orig:.1f})")
    print("-" * 75)

print("\n--- IMPACTO DO GATEKEEPER (COVARIATE SHIFT) ---\n")
print_funnel_report("Treino", data['y']['train'], y_train_gf)
print_funnel_report("Validação", data['y']['val'], y_val_gf)
print_funnel_report("Teste", data['y']['test'], y_test_gf)

## Discovery Model
Instanciamos a classe `GreatFilterModel` acoplando os `buffer_limits` para que o *Soft Buffer Training* avalie adequadamente a importância das features nas zonas limítrofes entre calmaria (Classe B) e alerta (Classe C).

In [ ]:
discovery_model = GreatFilterModel(
    buffer_limits=(1e-6, 1e-5), # Fronteira crítica de decisão B -> C
    buffer_weight=0.2,
    params={
        'n_estimators': 300,
        'learning_rate': 0.05,
        'max_depth': 5,
        'n_jobs': -1,
        'random_state': 42
    }
)

In [ ]:
selected_features = discovery_model.discover_top_features(
    x=X_train_gf,
    y=y_train_gf,
    flux_values=flux_train_gf,
    cumulative_threshold=0.95
)

In [ ]:
selected_features

## Hyperparameter Tuning (Optuna)
O espaço de busca do Optuna foi ajustado para permitir que o modelo transite entre tratar a distorção do Covariate Shift e proteger as instâncias raras (Flares M/X).

In [ ]:
print("Transferindo dados para a VRAM da GPU...")

# 1. Filtramos as features selecionadas AINDA NO PANDAS (na CPU)
X_train_filtered = X_train_gf[selected_features].astype('float32')
X_val_filtered = X_val_gf[selected_features].astype('float32')

# 2. Convertendo Arrays de Features e Targets para o CuPy
X_train_gpu = cp.array(X_train_filtered.values)
y_train_gpu = cp.array(y_train_gf.values.astype('float32'))

X_val_gpu = cp.array(X_val_filtered.values)
y_val_gpu = cp.array(y_val_gf.values.astype('float32'))

# 3. Convertendo Arrays de Fluxo (Necessário para o SoftBufferXGBModel injetar a física)
flux_train_gpu = cp.array(flux_train_gf.values.astype('float32'))
flux_val_gpu = cp.array(flux_val_gf.values.astype('float32'))

print("Transferência concluída. Dados alocados na GPU.")

In [ ]:
def objective(trial):
    # 1. Calcula a proporção exata de desbalanceamento no Treino filtrado (Covariate Shift)
    neg_count = (y_train_gf == 0).sum()
    pos_count = (y_train_gf == 1).sum()
    imbalance_ratio = neg_count / pos_count if pos_count > 0 else 1.0

    pruning_callback = XGBoostPruningCallback(trial, 'validation_0-aucpr')

    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'aucpr',
        'n_estimators': 1000,
        'random_state': 1502,
        'n_jobs': -1,
        'device': 'cuda',

        'early_stopping_rounds': 50,
        'callbacks': [pruning_callback],

        # Espaço de busca modificado: não restringimos o modelo a pesos extremamente baixos,
        # permitindo-lhe equilibrar o Covariate Shift sem destruir o recall das classes M e X
        'scale_pos_weight': trial.suggest_float("scale_pos_weight", imbalance_ratio * 0.5, 3.0),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'gamma': trial.suggest_float('gamma', 0.1, 5.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 25)
    }

    # O Great Filter precisa conhecer seus limites de buffer físico para inicializar
    model = GreatFilterModel(
        params=params, 
        buffer_limits=(1e-6, 1e-5), 
        buffer_weight=0.2,
        features_to_keep=None
    )

    # Injeção de 'flux_values' permite a calibração de pesos de amostra baseada em zonas de fluxo
    model.fit(
        x=X_train_gpu,
        y=y_train_gpu,
        flux_values=flux_train_gpu,
        eval_set=[(X_val_gpu, y_val_gpu)],
        verbose=False
    )

    y_pred_proba_raw = model.predict_proba(X_val_gpu)[:, 1]
    y_pred_proba_cpu = y_pred_proba_raw.get() if hasattr(y_pred_proba_raw, 'get') else y_pred_proba_raw

    pr_auc = average_precision_score(y_val_gf, y_pred_proba_cpu)

    return pr_auc

In [ ]:
study = optuna.create_study(direction='maximize')
print("\nIniciando tuning...")
study.optimize(objective, n_trials=500)

print(f"\nBest Score (PR AUC): {study.best_value:.4f}")
best_params = study.best_params

best_params.update({
    'n_estimators': 1000, 'objective': 'binary:logistic',
    'eval_metric': 'aucpr', 'random_state': 1502,
    'n_jobs': -1, 'early_stopping_rounds': 50
})

In [ ]:
final_model = GreatFilterModel(
    params=study.best_params, 
    buffer_limits=(1e-6, 1e-5),
    buffer_weight=0.2,
    features_to_keep=selected_features
)

# Treinamento final ancorado nos fluxos de sobreviência
final_model.fit(
    x=X_train_gf, 
    y=y_train_gf,
    flux_values=flux_train_gf,
    verbose=True
)

## Threshold Tuning

In [ ]:
fig = final_model.get_threshold_graph(X_val_gf, y_val_gf)
# display(fig)

In [ ]:
optimal_threshold = final_model.optimize_threshold(
    X_val_gf,
    y_val_gf,
    target_recall=0.95, # FILTRAGEM SEGURA: Tenta reter 95% das explosões C+ sobreviventes
    beta=1.5            # ÁRBITRO FLEXÍVEL: F1.5-Score. Dá um peso 1.5x maior para o Recall em relação à Precisão.
)

## Results

In [ ]:
# ==========================================
# 1. PRÉ-COMPUTAÇÃO DE VETORES (Faz a inferência apenas UMA vez)
# ==========================================
x_test_df = X_test_gf
y_true = y_test_gf.values.astype(int)
flux_test = flux_test_gf

y_prob = final_model.predict_proba(x_test_df)[:, 1]
y_pred = (y_prob >= optimal_threshold).astype(int)

# Dedução da Persistência: Se houve flare nas últimas 24h, preveja Alerta(1)
soma_flares_passado = x_test_df['count_C_24h'] + x_test_df['count_M_24h'] + x_test_df['count_X_24h']
y_persistence = (soma_flares_passado > 0).astype(int).values

In [ ]:
print("--- RELATÓRIO DE CLASSIFICAÇÃO ---")
print(final_model.get_classification_report(y_true, y_pred, target_names=['No Flare', 'Flare']))

In [ ]:
print("\n--- MÉTRICAS ABRANGENTES ---")
comprehensive_df = final_model.get_comprehensive_metrics(y_true, y_pred, y_prob)
display(comprehensive_df)

In [ ]:
print("\n--- PR-F1 (SKILL SCORE RELATIVO) ---")
pr_f1_score = final_model.calculate_prss(y_true, y_pred, y_persistence)
print(f"PR-F1 Score: {pr_f1_score:.4f}")

In [ ]:
print("\n--- ANÁLISE AC/NC (ACTIVITY CHANGE) ---")
ac_nc_df = final_model.analyze_ac_nc_performance(y_true, y_pred, y_persistence)
display(ac_nc_df)

In [ ]:
print("\n--- DISTRIBUIÇÃO DE ERROS POR CLASSE SOLAR ---")
error_dist_df = final_model.analyze_error_distribution(y_true, y_pred, flux_test)
display(error_dist_df)

In [ ]:
print("\n--- ANÁLISE DE FLUXO (ZONAS) ---")
fig_flux, summary_flux = final_model.analyze_flux_errors(y_true, y_pred, flux_test, buffer_limits=[1e-6, 1e-5])
display(summary_flux)
display(fig_flux)

## Features Importance

In [ ]:
print("\n--- IMPORTÂNCIA DAS FEATURES (GAIN) ---")
features_importance = final_model.get_feature_importance()
display(features_importance.head(10))

## Export

In [ ]:
SAVE_PATH = os.getenv('GLOBAL_XRAY_FINAL_MODELS_PATH')
os.makedirs(SAVE_PATH, exist_ok=True)

final_model.save(os.path.join(SAVE_PATH, 'great_filter_v1.joblib'))

print(f"Modelo Great Filter exportado com sucesso para: {SAVE_PATH}")
print(f"Threshold otimizado embutido: {final_model.threshold:.4f}")
print(f"Total de features retidas: {len(final_model.features_to_keep)}")